# Phase 1 - QLoRA SFT (Unsloth)

**Colab, A100.** Off-policy distillation: fine-tune the student to imitate the teacher's verified-correct trajectories (`docs/guide.v3.0.md` section 5.4).

Pipeline position: `filter.py` -> `format_sft.py` -> **`train.jsonl`** -> *this notebook* -> merged model -> `02_serve_vllm.ipynb` (serve + eval).

Run top to bottom. The harness (agent loop, tools, eval) stays frozen; only the weights change.

## 1. Install
Versions move fast. If an import below fails, follow the current Unsloth install docs (unsloth.ai/docs). `unsloth` pulls compatible `trl`, `peft`, `accelerate`, `bitsandbytes`, `transformers`, `datasets`. vLLM is **not** installed here - serving/eval lives in `02_serve_vllm.ipynb`, and keeping it out avoids dependency conflicts during training.

In [ ]:
!pip install -q unsloth

In [ ]:
# GPU check - make this a habit at the top of every notebook (guide 3.4).
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU (A100 for training).'

## 2. Config and paths
Mirrors `config.py`. Set `TRAIN_FILE` to wherever `train.jsonl` lands on Colab (next section).

In [ ]:
import os

MODEL_NAME     = 'Qwen/Qwen2.5-Coder-7B-Instruct'  # = config.STUDENT_MODEL
MAX_SEQ_LENGTH = 4096                              # = config.MAX_SEQ_LENGTH (from format_sft.py length report)
LORA_R         = 32
LORA_ALPHA     = 32
LORA_DROPOUT   = 0
TARGET_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

# Where the SFT data lives ON COLAB (see next section for how to get it here).
TRAIN_FILE  = '/content/train.jsonl'

# Outputs.
ADAPTER_DIR = '/content/qwen-sql-sft-lora'      # small LoRA adapter
MERGED_DIR  = '/content/qwen-sql-sft-merged'    # 16-bit merged model for vLLM (notebook 02)
DRIVE_DIR   = '/content/drive/MyDrive/text2sql-distill'  # persisted copy (if Drive mounted)

os.environ['WANDB_DISABLED'] = 'true'           # unattended run: never block on a wandb prompt

## 3. Get `train.jsonl` onto Colab
`data/sft/` is gitignored, so it does not ride along with a repo clone - move it deliberately by ONE of:
- **Upload:** Colab left panel (Files) > upload your local `data/sft/train.jsonl` to `/content`.
- **Drive:** run the mount cell, then point `TRAIN_FILE` at your Drive copy.
- **git:** `!git clone <your-repo>` and set `TRAIN_FILE` to `.../data/sft/train.jsonl` (only if you committed it).

The mount cell is optional - skip it if you uploaded the file directly.

In [ ]:
# Mount Drive. REQUIRED here - the final cell auto-persists outputs to Drive.
# (Also lets you load train.jsonl from Drive if you put it there.)
from google.colab import drive
drive.mount('/content/drive')
# If your data lives on Drive, repoint TRAIN_FILE, e.g.:
# TRAIN_FILE = '/content/drive/MyDrive/text2sql-distill/data/sft/train.jsonl'

In [ ]:
# Verify the data is present and non-empty before spending GPU time.
import pathlib
p = pathlib.Path(TRAIN_FILE)
assert p.is_file(), (
    f'train.jsonl not found at {TRAIN_FILE!r}. Get it onto Colab first (upload / Drive / git), '
    f'then set TRAIN_FILE.'
)
n = sum(1 for line in p.open(encoding='utf-8') if line.strip())
print(f'Found {n} SFT examples at {TRAIN_FILE}')
assert n > 0, 'train.jsonl is empty - did format_sft.py run on the filtered trajectories?'

## 4. Load the model (4-bit) and attach LoRA
Import `unsloth` first so it can patch `transformers`/`trl`. `load_in_4bit=True` is the QLoRA part: the 7B base is quantized to 4-bit, adapters stay in bf16.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,        # auto: bf16 on A100
    load_in_4bit   = True,        # QLoRA
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',   # long-context memory saver
    random_state               = 3407,
)

## 5. Load data and guard against silent truncation
The `{"messages": [...]}` lines are conversational data; TRL applies the tokenizer's chat template automatically at train time. Before training, render every example with that same template and confirm nothing exceeds `MAX_SEQ_LENGTH` - a truncated trajectory loses its final `submit`, corrupting the target (guide 5.3, trap 7). This also proves the template handles our tool-call / tool-result turns.

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files=TRAIN_FILE, split='train')
print('raw:', ds)

# Render each trajectory to ONE training string with the model's chat template.
# We do this explicitly instead of relying on TRL to auto-template the 'messages'
# column, because some Unsloth/TRL builds require a 'text' field (or a
# formatting_func) and will NOT auto-format conversational data -> the
# 'Unsloth: You must specify a formatting_func' error. This relies on the
# arguments-as-dict normalization that format_sft.py already did, so tool calls
# render correctly.
def render(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)}
ds = ds.map(render, remove_columns=ds.column_names)

# Truncation guard (guide 5.3, trap 7): token lengths as the trainer will see them.
lengths = sorted(len(tokenizer(t, add_special_tokens=False)['input_ids']) for t in ds['text'])
m = len(lengths)
pct = lambda q: lengths[min(m - 1, int(q * (m - 1)))]
over = sum(1 for L in lengths if L > MAX_SEQ_LENGTH)
print(f'tokens  min/p50/p95/p99/max = {lengths[0]}/{pct(.5)}/{pct(.95)}/{pct(.99)}/{lengths[-1]}')
print(f'> MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH}): {over} ({over / m:.1%})')
if over:
    print('WARNING: raise MAX_SEQ_LENGTH (and reload the model) or drop the long trajectories,')
    print('         otherwise their tails (the final submit) get truncated.')

In [ ]:
# Eyeball one rendered example - confirm tool calls and tool results look right.
print(ds[0]['text'][:2000])

## 6. Train
Hyperparameters per guide 5.4. Watch the loss: it should drop smoothly. **Loss down != accuracy up** - 2 epochs is a sane start; more on a few thousand examples overfits. (TRL/Unsloth APIs move fast; if an arg is rejected, reconcile with the version you installed.)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,                 # pre-rendered 'text' column (see data cell)
    args = SFTConfig(
        dataset_text_field          = 'text',   # train on the rendered text, no auto-templating
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,    # effective batch size 8
        warmup_steps                = 5,
        num_train_epochs            = 2,    # 1-3; watch for overfit
        learning_rate               = 2e-4,
        logging_steps               = 5,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'linear',
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        seed                        = 3407,
        output_dir                  = 'outputs',
        report_to                   = 'none',   # no wandb in an unattended run
    ),
)

In [ ]:
trainer_stats = trainer.train()

## 7. Save and auto-persist to Drive
Save the small adapter (to resume/merge later) and the merged 16-bit model (what vLLM serves in notebook 02) to local `/content`, then **automatically copy both to Drive** so a disconnect can't cost you the run (guide 3.4). The merged model is ~15 GB — if your Drive lacks the space, set `PERSIST_MERGED = False` and push it to the HF Hub instead.

In [ ]:
# Adapter (small) - enough to reload for more training or to merge later.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Merged 16-bit - what vLLM serves in 02_serve_vllm.ipynb (~15 GB on disk).
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print('saved adapter ->', ADAPTER_DIR)
print('saved merged  ->', MERGED_DIR)

In [ ]:
# Auto-persist outputs to Drive at the end of the run.
import shutil
assert os.path.isdir('/content/drive'), (
    'Drive not mounted - run the mount cell (section 3) first, then re-run THIS cell. '
    'Your model is still in memory and on /content, so nothing is lost.'
)
os.makedirs(DRIVE_DIR, exist_ok=True)

PERSIST_MERGED = True   # set False if your Drive lacks ~15 GB free (adapter alone is tiny)

# adapter (small) - always persisted
adst = os.path.join(DRIVE_DIR, os.path.basename(ADAPTER_DIR))
shutil.copytree(ADAPTER_DIR, adst, dirs_exist_ok=True)
print('persisted adapter ->', adst)

# merged 16-bit (~15 GB) - what notebook 02 serves with vLLM
if PERSIST_MERGED:
    mdst = os.path.join(DRIVE_DIR, os.path.basename(MERGED_DIR))
    shutil.copytree(MERGED_DIR, mdst, dirs_exist_ok=True)
    print('persisted merged  ->', mdst)
else:
    print('skipped merged (PERSIST_MERGED=False) - push to HF Hub or copy it manually for notebook 02')

print('done. Drive contents:', os.listdir(DRIVE_DIR))

## 8. Smoke test (optional)
Sanity-check that the tuned model emits a sensible first action. Real execution-accuracy eval happens in `02_serve_vllm.ipynb` via the frozen agent loop + `src/eval.py`.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {'role': 'system', 'content': 'You are a text-to-SQL agent. Use tools to explore, then call submit.'},
    {'role': 'user',   'content': 'Database: concert_singer\nQuestion: How many singers are there?'},
]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                       return_tensors='pt').to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False))

## Next
Open `02_serve_vllm.ipynb`, serve `MERGED_DIR` with vLLM (on the L4), run the frozen agent loop over Spider `dev.json`, and score with `src/eval.py --official`. Append the result to `results/eval_runs.jsonl` and compare to baseline (guide 5.5).